# Segment amplitude atlas — true segments, failure modes, and what to change

Explicit eigenvalues and **classical & quantum amplitudes** for every canonical
segment cluster, computed from the matrix. Goal: know exactly which segments sit
where relative to τ, so we can target algorithm changes. Extends the
single-event eigen-analysis (nb5/nb6).

Rule: $`A=(\gamma+\delta)I-C`$, solve $`A\mathbf x=\delta\mathbf1`$, active iff
$`x_i>\tau=0.35`$; the 1BQF keeps each eigenmode $`\propto f(\lambda)=\cos(\lambda t/2)`$,
$`t=\pi/(\gamma+\delta)`$ (notch zero at $`\lambda=\gamma+\delta=4`$). γ=3, δ=1, s=4.

In [1]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd
plt.rcParams.update({"figure.dpi":110,"font.size":11,"axes.grid":True,"grid.alpha":0.3})
from pathlib import Path
OUT=Path("/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Segment_level_studies/outputs/amplitude_atlas")
OUT.mkdir(parents=True, exist_ok=True)
G,D=3.0,1.0; S=G+D; T=np.pi/S; TAU=D/(D+G)+0.10
f=lambda L: np.cos(L*T/2)
r=(S-np.sqrt(S*S-4))/2          # 2 - sqrt(3)
def path(m): return S*np.eye(m)-(np.eye(m,k=1)+np.eye(m,k=-1))
def star(m):
    A=S*np.eye(m+1); A[0,1:]=A[1:,0]=-1; return A
def solve_cluster(A):
    A=np.array(A,float); n=A.shape[0]; b=D*np.ones(n)
    w,U=np.linalg.eigh(A); beta=U.T@b; xC=np.linalg.solve(A,b)
    xqr=U@(beta*f(w)); mask=xC>TAU
    if mask.any() and np.linalg.norm(xqr[mask])>0:
        xQ=np.abs(xqr)*(np.linalg.norm(xC[mask])/np.linalg.norm(xqr[mask]))
    else:                          # no classically-active segment: scale to full-cluster norm
        nq=np.linalg.norm(xqr); xQ=np.abs(xqr)*(np.linalg.norm(xC)/nq) if nq>0 else np.abs(xqr)
    nbad=int((np.abs(w-S)<1e-9).sum())
    return w,xC,xQ,nbad
print(f"s={S} tau={TAU} notch λ={S} r=2-√3={r:.4f}; semi-infinite bulk x→δ/(s-2)={D/(S-2)}")

s=4.0 tau=0.35 notch λ=4.0 r=2-√3=0.2679; semi-infinite bulk x→δ/(s-2)=0.5


## 1. True track chains — first/last vs middle (closed form)
A track over $`m+1`$ planes is a continuation **path** $`P_m`$ of $`m`$ segments. Its
matrix is tridiagonal, $`A=sI-P_m`$, with **exact** spectrum and solution:
$$
\lambda_k=s-2\cos\frac{k\pi}{m+1},\qquad
x_{\rm outer}(m)=\frac12-\frac{r+r^{m}}{2\,(1+r^{m+1})},\quad r=2-\sqrt3 .
$$
The **endpoints** (first/last segment) are always the lowest: an endpoint has only
**one** continuation neighbour, an interior segment has **two**, so endpoints are
less reinforced. Values (s=4):

In [2]:
rows=[]
for m in range(1,9):
    w,xC,xQ,nb=solve_cluster(path(m))
    xo=0.5-(r+r**m)/(2*(1+r**(m+1)))
    rows.append(dict(m=m, n_planes=m+1, lam_min=round(w.min(),3), lam_max=round(w.max(),3),
                     x_outer=round(xC[0],4), x_outer_cf=round(xo,4),
                     x_inner_max=round(xC.max(),4),
                     C_active=int((xC>TAU).sum()), Q_active=int((xQ>TAU).sum())))
ladder=pd.DataFrame(rows); display(ladder)
print("note: a real 5-plane TRACK is P4 -> outer 4/11=0.3636, inner 5/11=0.4545.")
print("eigenvalues are NEVER on the notch for even-length chains; odd-length chains have one middle eigenvalue at s.")

,m,n_planes,lam_min,lam_max,x_outer,x_outer_cf,x_inner_max,C_active,Q_active
0,1,2,4.000,4.000,0.2500,0.2500,0.2500,0,0
1,2,3,3.000,5.000,0.3333,0.3333,0.3333,0,0
2,3,4,2.586,5.414,0.3571,0.3571,0.4286,3,1
3,4,5,2.382,5.618,0.3636,0.3636,0.4545,4,2
4,5,6,2.268,5.732,0.3654,0.3654,0.4808,5,3
5,6,7,2.198,5.802,0.3659,0.3659,0.4878,6,4
6,7,8,2.152,5.848,0.3660,0.3660,0.4948,7,5
7,8,9,2.121,5.879,0.3660,0.3660,0.4967,8,6


note: a real 5-plane TRACK is P4 -> outer 4/11=0.3636, inner 5/11=0.4545.
eigenvalues are NEVER on the notch for even-length chains; odd-length chains have one middle eigenvalue at s.


## 2. The endpoint failure mode (quantum) — first/last segments are halved
Explicitly, the real track $`P_4`$: classically all four are above τ, but the 1-bit
filter **halves the endpoints** below τ — only the middle pair survives.

In [3]:
w,xC,xQ,nb=solve_cluster(path(4))
print("TRUE track P4:")
print("  eigenvalues      :", np.round(w,3))
print("  filter f(λ)       :", np.round(f(w),3))
print("  classical x       :", np.round(xC,4), " (4/11, 5/11, 5/11, 4/11)  active:", np.where(xC>TAU)[0].tolist())
print("  quantum   x       :", np.round(xQ,4), "                          active:", np.where(xQ>TAU)[0].tolist())
print(f"  -> ENDPOINTS (first/last) classical {xC[0]:.3f} -> quantum {xQ[0]:.3f} < τ={TAU}: LOST.")
print(f"  -> MIDDLE classical {xC[1]:.3f} -> quantum {xQ[1]:.3f} > τ: kept. Quantum track efficiency = 2/4.")

fig,ax=plt.subplots(figsize=(8,4.6))
mm=np.arange(1,9); xo=[solve_cluster(path(m))[1][0] for m in mm]; xi=[solve_cluster(path(m))[1].max() for m in mm]
xoQ=[solve_cluster(path(m))[2][0] for m in mm]; xiQ=[solve_cluster(path(m))[2].max() for m in mm]
ax.plot(mm,xo,'o-',color="#1b7837",label="outer (first/last)  classical")
ax.plot(mm,xi,'s-',color="#2166ac",label="inner (middle) max  classical")
ax.plot(mm,xoQ,'o--',color="#1b7837",alpha=.6,label="outer  quantum (halved)")
ax.plot(mm,xiQ,'s--',color="#2166ac",alpha=.6,label="inner  quantum")
ax.axhline(TAU,color="k",ls=":",lw=1.3,label="τ=0.35"); ax.axhline(D/(S-2),color="grey",ls=":",lw=1,label="bulk 0.5")
ax.axvline(4,color="orange",alpha=.3,lw=8)
ax.text(4,0.18,"real track\n(P4)",ha="center",fontsize=8,color="darkorange")
ax.set_xlabel("chain length m (segments)"); ax.set_ylabel("activation")
ax.set_title("Endpoint vs middle activation: quantum halves the endpoints below τ", fontweight="bold")
ax.legend(fontsize=8,loc="lower right")
fig.tight_layout()
for e,dp in (("pdf",600),("png",300)): fig.savefig(OUT/f"chain_amplitude_ladder.{e}",dpi=dp,bbox_inches="tight",facecolor="white")
plt.show(); print("saved chain_amplitude_ladder")

TRUE track P4:
  eigenvalues      : [2.382 3.382 4.618 5.618]
  filter f(λ)       : [ 0.593  0.24  -0.24  -0.593]
  classical x       : [0.3636 0.4545 0.4545 0.3636]  (4/11, 5/11, 5/11, 4/11)  active: [0, 1, 2, 3]
  quantum   x       : [0.2575 0.522  0.522  0.2575]                           active: [1, 2]
  -> ENDPOINTS (first/last) classical 0.364 -> quantum 0.258 < τ=0.35: LOST.
  -> MIDDLE classical 0.455 -> quantum 0.522 > τ: kept. Quantum track efficiency = 2/4.


saved chain_amplitude_ladder


## 3. The full failure-mode catalogue
Every canonical cluster, with explicit eigenvalues, classical & quantum amplitudes,
and the failure it causes.

In [4]:
CONFIGS=[
 ("TRUE track (P4)", path(4), "true"),
 ("TRUE 4-hit fragment (P3)", path(3), "true"),
 ("TRUE 3-hit fragment (P2)", path(2), "true"),
 ("TRUE 2-hit stub (P1)", path(1), "true"),
 ("FALSE isolated (P1)", path(1), "false"),
 ("FALSE pair (P2)", path(2), "false"),
 ("FALSE bridge/triple (P3)", path(3), "false"),
 ("FALSE hub K(1,3)", star(3), "false"),
 ("FALSE hub K(1,4)", star(4), "false"),
 ("FALSE hub K(1,5)", star(5), "false"),
]
cat=[]
for name,A,kind in CONFIGS:
    w,xC,xQ,nb=solve_cluster(A)
    cat.append(dict(config=name, kind=kind, n=A.shape[0], eig=str(np.round(np.unique(w.round(3)),3).tolist()),
                    n_bad=nb, x_classical=str(np.round(xC,3).tolist()), x_quantum=str(np.round(xQ,3).tolist()),
                    C_act=int((xC>TAU).sum()), Q_act=int((xQ>TAU).sum())))
catdf=pd.DataFrame(cat);
pd.set_option("display.max_colwidth",60); display(catdf)

,config,kind,n,eig,n_bad,x_classical,x_quantum,C_act,Q_act
0,TRUE track (P4),true,4,"[2.382, 3.382, 4.618, 5.618]",0,"[0.364, 0.455, 0.455, 0.364]","[0.258, 0.522, 0.522, 0.258]",4,2
1,TRUE 4-hit fragment (P3),true,3,"[2.586, 4.0, 5.414]",1,"[0.357, 0.429, 0.357]","[0.27, 0.541, 0.27]",3,1
2,TRUE 3-hit fragment (P2),true,2,"[3.0, 5.0]",0,"[0.333, 0.333]","[0.333, 0.333]",0,0
3,TRUE 2-hit stub (P1),true,1,[4.0],1,[0.25],[0.25],0,0
4,FALSE isolated (P1),false,1,[4.0],1,[0.25],[0.25],0,0
5,FALSE pair (P2),false,2,"[3.0, 5.0]",0,"[0.333, 0.333]","[0.333, 0.333]",0,0
6,FALSE bridge/triple (P3),false,3,"[2.586, 4.0, 5.414]",1,"[0.357, 0.429, 0.357]","[0.27, 0.541, 0.27]",3,1
7,"FALSE hub K(1,3)",false,4,"[2.268, 4.0, 5.732]",2,"[0.538, 0.385, 0.385, 0.385]","[0.742, 0.247, 0.247, 0.247]",4,1
8,"FALSE hub K(1,4)",false,5,"[2.0, 4.0, 6.0]",3,"[0.667, 0.417, 0.417, 0.417, 0.417]","[0.955, 0.239, 0.239, 0.239, 0.239]",5,1
9,"FALSE hub K(1,5)",false,6,"[1.764, 4.0, 6.236]",4,"[0.818, 0.455, 0.455, 0.455, 0.455, 0.455]","[1.191, 0.238, 0.238, 0.238, 0.238, 0.238]",6,1


## 4. The two structural failures (why a flat threshold can't win)

In [5]:
# (a) degeneracy: a true 4-hit fragment (P3) and a false bridge (P3) are the SAME matrix
print("TRUE 4-hit fragment vs FALSE triple bridge — identical P3 matrices:")
print("  both: eig", np.round(np.linalg.eigvalsh(path(3)),3), " classical x", np.round(np.linalg.solve(path(3),D*np.ones(3)),3))
print("  -> NOT separable at the segment level: same eigenvalues, same amplitudes.\n")
# (b) the atlas number-line: all canonical segment activations vs tau
fig,ax=plt.subplots(2,1,figsize=(12,5.2),sharex=True)
ymap={"true":"#1b7837","false":"#c51b7d"}
for k,(name,A,kind) in enumerate(CONFIGS):
    w,xC,xQ,nb=solve_cluster(A)
    for sol,a in [(xC,ax[0]),(xQ,ax[1])]:
        a.scatter(np.unique(sol.round(3)), [k]*len(np.unique(sol.round(3))), s=60, color=ymap[kind], ec="k", lw=.4, zorder=3)
for a,lab in [(ax[0],"classical"),(ax[1],"quantum (1BQF)")]:
    a.axvline(TAU,color="k",ls="--",lw=1.4); a.set_yticks(range(len(CONFIGS)))
    a.set_yticklabels([c[0] for c in CONFIGS], fontsize=8)
    a.set_ylabel(lab,fontweight="bold"); a.set_xlim(-0.05,1.0)
ax[1].set_xlabel("segment activation"); ax[0].axvline(TAU,color="k",ls="--",lw=1.4,label="τ=0.35"); ax[0].legend(fontsize=8,loc="lower right")
ax[0].set_title("Amplitude atlas: distinct activation levels per cluster (green=true, pink=false; right of τ = active)",fontweight="bold",fontsize=10)
fig.tight_layout()
for e,dp in (("pdf",600),("png",300)): fig.savefig(OUT/f"amplitude_atlas.{e}",dpi=dp,bbox_inches="tight",facecolor="white")
plt.show(); print("saved amplitude_atlas")

TRUE 4-hit fragment vs FALSE triple bridge — identical P3 matrices:
  both: eig [2.586 4.    5.414]  classical x [0.357 0.429 0.357]
  -> NOT separable at the segment level: same eigenvalues, same amplitudes.



saved amplitude_atlas


## 5. Reading the atlas — and where to modify the algorithm

**True-segment levels (classical):** isolated stub 0.25 · pair 1/3=0.333 · triple
outer 5/14=0.357 · **track (P4) outer 4/11=0.364, inner 5/11=0.455** · → bulk 0.5.
Endpoints are the weakest true segments and sit **just** above τ.

**Failure modes:**

| # | mode | where | effect | classical | quantum |
|---|---|---|---|---|---|
| F1 | **endpoint halving** | true outer 0.364 → 0.258 | lose first/last segment of every track (eff −50% on clean P4) | ok | **fails** |
| F2 | **fragment loss** | true P2 0.333 < τ | hit-drop fragments lost | fails | (rescaled up, edge case) |
| F3 | **true/false degeneracy** | true P3 ≡ false P3 | bridges indistinguishable from track fragments | both | both |
| F4 | **false bridges (m≥3)** | false triple outer 0.357 > τ | false positives | fails | partial |
| F5 | **false hubs K(1,m)** | centre 0.54–0.67 (q 0.74–0.96) | strong false positives, m−1 bad eigenvalues | fails | **fails hard** |

**Implications for modifying the algorithm:**

1. **Endpoint-aware threshold (F1).** Endpoints are *structurally* lower (one
   neighbour, not two) — a single flat τ punishes them. A degree-/position-aware
   threshold (lower τ for boundary segments) or a boundary term in $`A`$ that lifts
   endpoints to the bulk level would recover them. The quantum halving makes this
   essential on the 1BQF.
2. **Don't fight F3 at the segment level.** A true 4-hit fragment and a false
   bridge are the *same* matrix — no segment-level cut separates them. This needs
   **track-level** (connected-component / fit) reasoning, or hit-occupancy
   constraints — *not* a better threshold.
3. **Hubs (F5)** are the cleanest target: they have a distinctive signature (high
   centre amplitude + several bad/notch eigenvalues). A per-hit occupancy penalty
   suppresses them — but the bifurcation study (`../Bifurification/`) shows the
   naive fork term breaks the 1BQF, so any such term must keep the false
   population pinned on the notch.
4. **The quantum filter** itself is the lever: $`f(\lambda)=\cos(\lambda t/2)`$ halves
   the off-notch true modes that build the endpoints. A higher-resolution
   (more-bit) inversion, or a tuned evolution time $`t`$, would reshape which true
   modes survive — directly addressing F1.